# ViTrueVoice — Pipeline Huấn Luyện SSL-AASIST & Đánh Giá Đối Chứng H1/H2 Trên Kaggle GPU

**Mục tiêu thí nghiệm (Must #2 - W5)**:
1. **Chạy thử nhanh (Smoke Test)**: 100 mẫu, 1 epoch để xác nhận đường ống, tải mô hình và đường dẫn âm thanh.
2. **Huấn luyện 3 nhánh thí nghiệm chính thức**:
   - **Nhánh A (Không tăng cường)**: `mode="clean"` (10.699 mẫu), không transform, Phase 1 = 4 epochs, Phase 2 = 60 epochs (nhân 3 để cân bằng optimizer steps).
   - **Nhánh B0 (Chỉ nén Codec, không RawBoost)**: `mode="augmented"` (31.327 mẫu), không transform, Phase 1 = 4 epochs, Phase 2 = 20 epochs.
   - **Nhánh B (ViTrueVoice Đầy đủ: Codec + RawBoost)**: `mode="augmented"` (31.327 mẫu), `make_rawboost()`, Phase 1 = 4 epochs, Phase 2 = 20 epochs.
3. **Đánh giá tự động trên Test In-domain và Test OOD** qua `scripts/eval_ssl_aasist.py`.
4. **Kiểm chứng giả thuyết khoa học** qua `scripts/compare_h1.py`:
   - **H1**: Giảm $\ge 30\%$ EER tương đối trên kênh nén (G.711 A-law, AMR-NB) so với Mô hình A.
   - **H2**: EER $\le 15\%$ trên công cụ TTS chưa từng thấy (`VieNeu-TTS`).
   - **Phân rã biến**: So sánh B0 vs B để bóc tách chính xác đóng góp của RawBoost (đọc `delta_rel_pct` và `ci_overlap`).


In [ ]:
# 1. Cài đặt phụ thuộc & Kiểm tra GPU
import subprocess
import sys
import torch

print('📦 Đang kiểm tra & cài đặt thư viện phụ thuộc...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'librosa', 'transformers'], check=False)

print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Không có GPU! Huấn luyện SSL-AASIST trên CPU sẽ rất chậm.')
print('=' * 60)


In [ ]:
# 2. Khởi tạo mã nguồn ViTrueVoice trên Kaggle container
import base64
import zlib
import json
import sys
from pathlib import Path

PKG_B64 = '...'

pkg_data = json.loads(zlib.decompress(base64.b64decode(PKG_B64.encode('ascii'))).decode('utf-8'))
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

for rel_path, content in pkg_data.items():
    file_path = working_dir / rel_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding='utf-8')

if str(working_dir) not in sys.path:
    sys.path.insert(0, str(working_dir))

print(f'✅ Đã khởi tạo thành công {len(pkg_data)} files mã nguồn tại: {working_dir}')


In [ ]:
# 3. Định vị dữ liệu âm thanh, tệp Manifest và Kiểm tra đường dẫn âm thanh
import zipfile
import pandas as pd
from pathlib import Path

working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

# Kiểm tra nếu dữ liệu đã tồn tại sẵn
candidates = [
    working_dir / 'data',
    Path('/kaggle/input/vietrue-model-training/data'),
    Path('/kaggle/input/vitruevoice-data-processed/data'),
    Path('/kaggle/input/vietruevoice-processed/data'),
    Path('./data'),
]

data_dir = None
for c in candidates:
    if c.exists() and (c / 'manifests' / 'train.csv').exists():
        data_dir = c
        break

if not data_dir:
    # Tìm file ZIP đã nén từ kernel sinh dữ liệu vietrue-model-training
    zip_candidates = (
        list(Path('/kaggle/input').rglob('*vitruevoice*.zip')) +
        list(working_dir.glob('*vitruevoice*.zip')) +
        list(Path('out').glob('*vitruevoice*.zip'))
    )
    if zip_candidates:
        zip_path = zip_candidates[0]
        print(f'📦 Tìm thấy file nén dữ liệu: {zip_path}. Đang giải nén...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(working_dir)
        data_dir = working_dir / 'data'
        print(f'✅ Đã giải nén vào {data_dir}')

if data_dir and (data_dir / 'manifests' / 'train.csv').exists():
    manifest_dir = data_dir / 'manifests'
    audio_root = working_dir
else:
    # Fallback kiểm tra thư mục manifests trực tiếp
    manifest_dir = Path('data/manifests') if Path('data/manifests/train.csv').exists() else Path('out')
    audio_root = Path('.')

print(f'📂 Manifest Directory: {manifest_dir}')
print(f'🎵 Audio Root        : {audio_root}')
assert (manifest_dir / 'train.csv').exists(), f'❌ Không tìm thấy train.csv tại {manifest_dir}!'
assert (manifest_dir / 'dev.csv').exists(), f'❌ Không tìm thấy dev.csv tại {manifest_dir}!'
assert (manifest_dir / 'test_indomain.csv').exists(), f'❌ Không tìm thấy test_indomain.csv tại {manifest_dir}!'

# Kiểm tra xác thực đường dẫn file âm thanh mẫu thực tế
train_sample_df = pd.read_csv(manifest_dir / 'train.csv', nrows=5)
sample_rel_path = train_sample_df['audio_path'].iloc[0]
sample_full_path = audio_root / sample_rel_path
print(f'🔍 Kiểm tra file âm thanh mẫu: {sample_full_path}')
assert sample_full_path.exists(), f'❌ Không tìm thấy file âm thanh mẫu tại {sample_full_path}!'
print('✅ Đường dẫn âm thanh mẫu hoàn toàn hợp lệ!')


In [ ]:
# 4. Hàm thực thi subprocess dùng chung (kế thừa PYTHONPATH và log realtime)
import os
import subprocess
import sys
from pathlib import Path

WORK = Path('/kaggle/working').resolve() if Path('/kaggle/working').exists() else Path('.').resolve()
ENV = {
    **os.environ,
    'PYTHONPATH': f"{WORK}:{os.environ.get('PYTHONPATH', '')}",
    'PYTHONUNBUFFERED': '1',
}

def run(cmd, name):
    """Chạy lệnh từ thư mục gốc dự án, có PYTHONPATH, log hiện theo thời gian thực."""
    print(f'\n{"="*75}\n🚀 {name}\nLệnh: {" ".join(map(str, cmd))}\n{"="*75}', flush=True)
    res = subprocess.run([str(c) for c in cmd], cwd=WORK, env=ENV)
    assert res.returncode == 0, f'❌ {name} thất bại (mã {res.returncode})'

print('✅ Đã khởi tạo hàm run() với PYTHONPATH =', ENV['PYTHONPATH'])


In [ ]:
# 5. Kiểm thử nhanh (Smoke Test) xác nhận toàn bộ đường ống trước khi chạy chính thức
# Chạy 100 mẫu, Phase 1 = 1 epoch, Phase 2 = 0 epoch
run([
    sys.executable, '-u', 'scripts/train.py',
    '--branch', 'A',
    '--manifest-dir', str(manifest_dir),
    '--audio-root', str(audio_root),
    '--out-dir', 'out/smoke_test',
    '--batch-size', '16',
    '--limit', '100',
    '--phase1-epochs', '1',
    '--phase2-epochs', '0',
], 'Smoke Test (100 mẫu, 1 epoch)')


In [ ]:
# 6. Huấn luyện Mô hình A (Không tăng cường - Baseline SSL-AASIST)
# mode='clean' (10.699 mẫu sạch), transform=None, Phase 1 = 4 ep, Phase 2 = 60 ep
run([
    sys.executable, '-u', 'scripts/train.py',
    '--branch', 'A',
    '--manifest-dir', str(manifest_dir),
    '--audio-root', str(audio_root),
    '--out-dir', 'out/run_A',
    '--batch-size', '16',
], 'Huấn luyện Nhánh A (Clean Baseline)')


In [ ]:
# 7. Huấn luyện Mô hình B0 (Chỉ nén Codec, không RawBoost)
# mode='augmented' (31.327 mẫu), transform=None, Phase 1 = 4 ep, Phase 2 = 20 ep
run([
    sys.executable, '-u', 'scripts/train.py',
    '--branch', 'B0',
    '--manifest-dir', str(manifest_dir),
    '--audio-root', str(audio_root),
    '--out-dir', 'out/run_B0',
    '--batch-size', '16',
], 'Huấn luyện Nhánh B0 (Codec only)')


In [ ]:
# 8. Huấn luyện Mô hình B (ViTrueVoice Đầy Đủ: Codec + RawBoost Online)
# mode='augmented' (31.327 mẫu), transform=make_rawboost(), Phase 1 = 4 ep, Phase 2 = 20 ep
run([
    sys.executable, '-u', 'scripts/train.py',
    '--branch', 'B',
    '--manifest-dir', str(manifest_dir),
    '--audio-root', str(audio_root),
    '--out-dir', 'out/run_B',
    '--batch-size', '16',
], 'Huấn luyện Nhánh B (ViTrueVoice Đầy Đủ)')


In [ ]:
# 9. Đánh giá tự động các mô hình trên Test Sets (In-domain & OOD)
manifest_args = [f'in_domain={manifest_dir}/test_indomain.csv']
if (manifest_dir / 'test_ood.csv').exists():
    manifest_args.append(f'ood={manifest_dir}/test_ood.csv')
    print('ℹ️ Đã phát hiện và bổ sung test_ood.csv vào danh sách đánh giá.')
else:
    print('⚠️ Lưu ý: Chưa phát hiện test_ood.csv, đánh giá H2 sẽ dựa trên test_indomain.csv.')

for branch in ['A', 'B0', 'B']:
    ckpt = f'out/run_{branch}/best_model.pt'
    if Path(ckpt).exists():
        run([
            sys.executable, '-u', 'scripts/eval_ssl_aasist.py',
            '--ckpt', ckpt,
            '--manifest', *manifest_args,
            '--audio-root', str(audio_root),
            '--role', 'eval',
            '--out', f'out/eval_metrics_{branch}.csv',
            '--scores-out', f'out/scores_{branch}.csv',
        ], f'Chấm điểm mô hình {branch}')
    else:
        print(f'⚠️ Checkpoint không tồn tại: {ckpt}')


In [ ]:
# 10. Kiểm chứng giả thuyết khoa học H1 & H2
# 10a. Đối chứng chính thức H1 & H2 giữa Mô hình A (Không tăng cường) và B (ViTrueVoice)
if Path('out/eval_metrics_A.csv').exists() and Path('out/eval_metrics_B.csv').exists():
    run([
        sys.executable, '-u', 'scripts/compare_h1.py',
        '--a', 'out/eval_metrics_A.csv',
        '--b', 'out/eval_metrics_B.csv',
        '--out-dir', 'out/comparison_A_B',
    ], 'H1/H2 A vs B')

# 10b. Đối chứng bóc tách đóng góp của RawBoost (B0 vs B)
# Lưu ý: Đọc delta_rel_pct và ci_overlap để đo lường RawBoost, không trích verdict H1 từ cặp này.
if Path('out/eval_metrics_B0.csv').exists() and Path('out/eval_metrics_B.csv').exists():
    run([
        sys.executable, '-u', 'scripts/compare_h1.py',
        '--a', 'out/eval_metrics_B0.csv',
        '--b', 'out/eval_metrics_B.csv',
        '--out-dir', 'out/comparison_B0_B',
    ], 'B0 vs B (Phân rã RawBoost: đọc delta_rel_pct & ci_overlap)')


In [ ]:
# 11. Hiển thị báo cáo tóm tắt Markdown trên Notebook
from IPython.display import display, Markdown

rep_ab = Path('out/comparison_A_B/h1_h2_summary.md')
if rep_ab.exists():
    print('=' * 80)
    print('📌 KẾT QUẢ ĐỐI CHỨNG GIẢ THUYẾT H1 & H2 (MÔ HÌNH A vs MÔ HÌNH B):')
    print('=' * 80)
    display(Markdown(rep_ab.read_text(encoding='utf-8')))

rep_b0b = Path('out/comparison_B0_B/h1_h2_summary.md')
if rep_b0b.exists():
    print('\n' + '=' * 80)
    print('📌 PHÂN RÃ ĐÓNG GÓP RIÊNG BIỆT CỦA RAWBOOST (MÔ HÌNH B0 vs MÔ HÌNH B):')
    print('=' * 80)
    display(Markdown(rep_b0b.read_text(encoding='utf-8')))


In [ ]:
# 12. Đóng gói kết quả đầu ra cho Kaggle Output
import shutil
from pathlib import Path

working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
out_dir = Path('out')

# Copy các file báo cáo và metrics ra root working_dir để dễ tải
for f in out_dir.rglob('*.csv'):
    shutil.copy(f, working_dir / f.name)
for f in out_dir.rglob('*.md'):
    shutil.copy(f, working_dir / f.name)

for branch in ['A', 'B0', 'B']:
    ckpt = out_dir / f'run_{branch}' / 'best_model.pt'
    if ckpt.exists():
        shutil.copy(ckpt, working_dir / f'best_model_{branch}.pt')

print('🎉 ĐÃ HOÀN TẤT TOÀN BỘ PIPELINE HUẤN LUYỆN VÀ ĐÁNH GIÁ ĐỐI CHỨNG!')
print('Tải kết quả về máy local: python manage_kaggle.py output')
